# Dynamic Nelson-Siegel con Kalman\n\nNotebook base para validar la lógica de la pestaña `Kalman` de la app estática. La idea es usar un Nelson-Siegel estándar como ecuación de medición y una transición AR(1) diagonal para `level`, `slope` y `curvature`.

In [ ]:
from pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nDATA_PATH = Path('..') / '..' / 'api_js' / 'data' / 'market_rates.csv'\ndf = pd.read_csv(DATA_PATH)\ndf['Date'] = pd.to_datetime(df['Date'])\ndf.head()

In [ ]:
RATE_MONTHS = {\n    'TPM': 1,\n    'SPC_03Y': 3,\n    'SPC_06Y': 6,\n    'SPC_1Y': 12,\n    'SPC_2Y': 24,\n    'SPC_3Y': 36,\n    'SPC_4Y': 48,\n    'SPC_5Y': 60,\n    'SPC_10Y': 120,\n}\nCOLUMNS = list(RATE_MONTHS.keys())\nLAMBDA = 0.0609\n\nclean = df[['Date', *COLUMNS]].dropna().copy()\ntaus = np.array([RATE_MONTHS[c] / 12 for c in COLUMNS], dtype=float)\n\ndef ns_loadings(taus, lam):\n    slope = (1 - np.exp(-lam * taus)) / (lam * taus)\n    curvature = slope - np.exp(-lam * taus)\n    return np.column_stack([np.ones_like(taus), slope, curvature])\n\nH = ns_loadings(taus, LAMBDA)\nclean.shape

In [ ]:
yields = clean[COLUMNS].to_numpy(dtype=float)\nols_betas = np.vstack([np.linalg.lstsq(H, y, rcond=None)[0] for y in yields])\nols = pd.DataFrame(ols_betas, columns=['level', 'slope', 'curvature'])\nols.index = clean['Date']\nols.head()

In [ ]:
def fit_ar1(x):\n    x0 = x[:-1]\n    x1 = x[1:]\n    X = np.column_stack([np.ones_like(x0), x0])\n    beta = np.linalg.lstsq(X, x1, rcond=None)[0]\n    resid = x1 - X @ beta\n    return beta[0], beta[1], resid.var(ddof=1)\n\ntransitions = np.array([fit_ar1(ols[c].to_numpy()) for c in ols.columns])\ntransitions

In [ ]:
F = np.diag(transitions[:, 1])\nc = transitions[:, 0]\nQ = np.diag(np.maximum(transitions[:, 2], 1e-6))\nresiduals = yields - np.vstack([H @ b for b in ols_betas])\nR = np.diag(np.maximum(residuals.var(axis=0, ddof=1), 1e-6))\n\na = ols_betas[0].copy()\nP = np.diag(np.maximum(ols.var(ddof=1).to_numpy(), 1e-4))\nI = np.eye(3)\n\nfiltered = []\nfor y in yields:\n    a_pred = c + F @ a\n    P_pred = F @ P @ F.T + Q\n    v = y - H @ a_pred\n    S = H @ P_pred @ H.T + R\n    K = P_pred @ H.T @ np.linalg.inv(S)\n    a = a_pred + K @ v\n    P = (I - K @ H) @ P_pred\n    filtered.append(a.copy())\n\nfiltered = pd.DataFrame(filtered, columns=['level', 'slope', 'curvature'], index=clean['Date'])\nfiltered.tail()

In [ ]:
curve_months = np.arange(1, 121)\ncurve_taus = curve_months / 12\nH_curve = ns_loadings(curve_taus, LAMBDA)\nlast_beta = filtered.iloc[-1].to_numpy()\ncurve = H_curve @ last_beta\npd.DataFrame({'MaturityMonths': curve_months, 'Rate': curve}).head()

## Lectura\n\n- `ols_betas` entrega la referencia estática fecha a fecha.\n- `filtered` entrega factores suavizados temporalmente.\n- `curve` reconstruye la curva filtrada de la fecha más reciente.\n\nEsta es la misma aproximación simplificada que usa la pestaña `Kalman` de la app JS: medición Nelson-Siegel, transición AR(1) diagonal y reconstrucción de curva spot/forward desde los estados latentes.